# Preparing UChicago DM Data for BEHRT

This notebook transforms the **UChicago Diabetes Mellitus (DM)** cohort data into the format
required by the **BEHRT** model (BERT for Electronic Health Records). We focus exclusively
on two source files:

- **`Anony DM Demographics.csv`** — one row per patient with demographics and cohort dates
- **`Anony DM Diagnoses Final Cohort with Inpt and Outpt.csv`** — longitudinal diagnosis records (ICD-9/10)

---

## What BEHRT Expects

BEHRT consumes **per-patient sequences** with these parallel lists:

| Field | Type | Description |
|-------|------|-------------|
| `code` | `list[str]` | Time-sorted diagnosis tokens, with `'SEP'` separating visits |
| `age`  | `list[str]` | Patient age (as string) at each token position |

Plus three vocabulary/dictionary artifacts:
- **`token2idx`** — maps each diagnosis code + special tokens (`PAD`, `CLS`, `SEP`, `MASK`, `UNK`) to an integer
- **`age2idx`** — maps age strings (0–119) + special tokens (`PAD`, `UNK`) to integers
- **`sex_dict`** — maps patient ID → sex integer (1=male, 2=female)

---

## Key Mapping Decisions

| UChicago Field | BEHRT Field | Transformation |
|----------------|-------------|----------------|
| `studyid` | `enrolid` | Rename (simple column rename) |
| `icd10` / `icd9` | `code` tokens (PheCodes) | Map ICD-10 and ICD-9 to PheCode phenotype strings using the PheTK package (PheCode X) |
| `contact_date` | visit day | Parse to date; group diagnoses by date |
| `age_start_dat` + `start_dat` | `age` per visit | Derive birth year; compute age at each contact |
| `sex` (string) | `sex_dict` (int) | `'male'→1`, `'female'→2` |

### Why PheCodes?

BEHRT's original pipeline uses **phenotype strings** (e.g., "Fracture of femur") as
tokens — not raw ICD codes. PheCodes map both ICD-9 and ICD-10 to a shared set of
~3,500 clinically meaningful phenotype categories, which:
1. **Unifies** ICD-9 and ICD-10 — the same clinical concept gets the same token regardless of coding era
2. **Reduces vocabulary** — from ~20k raw ICD codes to ~3.5k phenotypes, improving embedding quality
3. **Matches the BEHRT design** — the model was built around phenotype-level tokens

---
## Step 1: Load and Inspect the Raw Data

We start by loading both CSV files and examining their shapes, column types,
and a handful of sample rows. This gives us a baseline understanding before
any transformations.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

DATA_DIR = Path('data/uchicago')
BEHRT_DIR = Path('data/behrt')
OUTPUT_DIR = Path('data/behrt/uchicago_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load Demographics ──
demo_file = DATA_DIR / 'Anony DM Demographics.csv'
demo = pd.read_csv(demo_file)
print(f'Demographics: {demo.shape[0]:,} patients, {demo.shape[1]} columns')
print(f'Columns: {list(demo.columns)}\n')
display(demo.head(3))

In [ ]:
# ── Load Diagnoses ──
diag_file = DATA_DIR / 'Anony DM Diagnoses Final Cohort with Inpt and Outpt.csv'
diag = pd.read_csv(diag_file)
print(f'Diagnoses: {diag.shape[0]:,} rows, {diag.shape[1]} columns')
print(f'Columns: {list(diag.columns)}\n')
display(diag.head(3))

---
## Step 2: Data Quality Assessment

Before transforming anything, we check for:
- Missing values in critical columns
- Patient ID overlap between the two files
- Date format consistency
- ICD code coverage (ICD-10 vs ICD-9)

In [ ]:
# ── 2a. Missing values in Demographics ──
# These are the columns we need for BEHRT: studyid, sex, age_start_dat, start_dat
critical_demo_cols = ['studyid', 'sex', 'age_start_dat', 'start_dat']
print('=== Demographics: missing values in critical columns ===')
for col in critical_demo_cols:
    n_missing = demo[col].isna().sum()
    pct = 100 * n_missing / len(demo)
    print(f'  {col:20s}: {n_missing:>6,} missing ({pct:.2f}%)')

print(f'\n=== Demographics: sex distribution ===')
print(demo['sex'].value_counts())

In [ ]:
# ── 2b. Missing values in Diagnoses ──
# The critical columns: studyid, contact_date, icd10
critical_diag_cols = ['studyid', 'contact_date', 'icd10', 'icd9']
print('=== Diagnoses: missing values in critical columns ===')
for col in critical_diag_cols:
    n_missing = diag[col].isna().sum()
    pct = 100 * n_missing / len(diag)
    print(f'  {col:20s}: {n_missing:>6,} missing ({pct:.2f}%)')

print(f'\n=== ICD code coverage ===')
has_icd10 = diag['icd10'].notna().sum()
has_icd9 = diag['icd9'].notna().sum()
has_both = (diag['icd10'].notna() & diag['icd9'].notna()).sum()
has_neither = (diag['icd10'].isna() & diag['icd9'].isna()).sum()
print(f'  Has ICD-10:          {has_icd10:>10,} ({100*has_icd10/len(diag):.1f}%)')
print(f'  Has ICD-9:           {has_icd9:>10,} ({100*has_icd9/len(diag):.1f}%)')
print(f'  Has both:            {has_both:>10,} ({100*has_both/len(diag):.1f}%)')
print(f'  Has neither (gap):   {has_neither:>10,} ({100*has_neither/len(diag):.1f}%)')

In [ ]:
# ── 2c. Patient ID overlap between Demographics and Diagnoses ──
demo_ids = set(demo['studyid'].unique())
diag_ids = set(diag['studyid'].unique())

print(f'Unique patients in Demographics: {len(demo_ids):,}')
print(f'Unique patients in Diagnoses:    {len(diag_ids):,}')
print(f'Overlap (in both):               {len(demo_ids & diag_ids):,}')
print(f'In Demographics only:            {len(demo_ids - diag_ids):,}')
print(f'In Diagnoses only:               {len(diag_ids - demo_ids):,}')

In [ ]:
# ── 2d. Date format inspection ──
# contact_date looks like '2015-02-19 00:00:00.000' (ISO datetime)
# start_dat in demographics looks like '21aug2013' (ddmonyyyy)
print('=== Sample dates ===')
print(f'  Diagnoses contact_date samples: {diag["contact_date"].dropna().head(3).tolist()}')
print(f'  Demographics start_dat samples: {demo["start_dat"].dropna().head(3).tolist()}')
print(f'  Demographics dm_min_dat samples: {demo["dm_min_dat"].dropna().head(3).tolist()}')

---
## Step 3: Parse Dates and Derive Birth Year

BEHRT needs the patient's **age at each visit**. UChicago provides:
- `age_start_dat` — the patient's age (fractional years) at `start_dat`
- `start_dat` — a reference date in `ddmonyyyy` format (e.g., `21aug2013`)

From these, we derive:
```
birth_year = year_of(start_dat) - int(age_start_dat)
```

Then for each diagnosis row, age = `year_of(contact_date) - birth_year`.

In [ ]:
# ── 3a. Parse the demographics start_dat (ddmonyyyy format) ──
demo['start_dat_parsed'] = pd.to_datetime(demo['start_dat'], format='%d%b%Y', errors='coerce')

n_failed = demo['start_dat_parsed'].isna().sum()
print(f'Parsed start_dat: {n_failed} failures out of {len(demo):,}')

# ── 3b. Derive approximate birth year ──
# age_start_dat is a float (e.g., 69.71), so birth_year = start_year - int(age)
demo['birth_year'] = demo['start_dat_parsed'].dt.year - demo['age_start_dat'].astype(int)

print(f'\nBirth year range: {demo["birth_year"].min():.0f} – {demo["birth_year"].max():.0f}')
print(f'Age at start_dat range: {demo["age_start_dat"].min():.1f} – {demo["age_start_dat"].max():.1f}')

display(demo[['studyid', 'start_dat', 'start_dat_parsed', 'age_start_dat', 'birth_year']].head(5))

---
## Step 4: Map ICD Codes to PheCode Phenotype Strings using PheTK

This is the critical step that makes UChicago data BEHRT-compatible. BEHRT's original
pipeline uses **phenotype strings** (e.g., "Fracture of femur") as tokens — not raw
ICD codes. We use the **PheTK** package (PheCode Toolkit) to map **both ICD-9 and
ICD-10** to PheCode X phenotype categories.

**Why this matters:**
- ICD-9 `714.0` ("Rheumatoid arthritis") and ICD-10 `M06.9` ("Rheumatoid arthritis, unspecified")
  both map to the **same PheCode phenotype**, so the model sees them as the same clinical concept
- This reduces the vocabulary from ~20k raw ICD codes to ~3.5k meaningful phenotypes
- Both ICD-9 and ICD-10 rows are treated as **real diagnoses** with equal weight

**PheTK workflow:**
1. Parse `contact_date` into datetime
2. Reshape UChicago ICD data into PheTK's expected format (`person_id`, `ICD`, `vocabulary_id`)
3. Initialize `PheTK.Phecode` with `platform="custom"` to load and validate the ICD data
4. Load PheTK's built-in PheCode X mapping table (`phecodeX.csv` — 156k ICD→PheCode entries)
5. Join the mapping at the **encounter level** to get a phenotype string per diagnosis row
6. Rows that don't map to any PheCode (mostly Z-codes, IMO internal codes) are dropped

In [ ]:
# ── 4a. Parse dates ──
# The data has two date columns:
#   - contact_date: ISO datetime format, but NaN for ~800k rows (24%)
#   - contact_dat:  ddmonyyyy format (e.g. "19feb2015"), never missing
# We use contact_dat as the primary source since it has zero missing values,
# and fall back to contact_date only if contact_dat were ever missing.

diag['contact_dt'] = pd.to_datetime(diag['contact_dat'], format='%d%b%Y', errors='coerce')

# If any contact_dat failed, try contact_date as fallback
n_missing = diag['contact_dt'].isna().sum()
if n_missing > 0:
    fallback = pd.to_datetime(diag['contact_date'], errors='coerce')
    diag['contact_dt'] = diag['contact_dt'].fillna(fallback)
    n_still_missing = diag['contact_dt'].isna().sum()
    print(f'contact_dat had {n_missing:,} parse failures; after contact_date fallback: {n_still_missing:,} remain')
else:
    print(f'Parsed all {len(diag):,} rows via contact_dat — zero failures')

# ── 4b. Reshape UChicago ICD data into PheTK's expected format ──
# PheTK expects: person_id, ICD, vocabulary_id ("ICD9CM" or "ICD10CM")
# UChicago has separate icd10 and icd9 columns per row. We melt them into
# long format so each ICD code gets its own row with the correct vocabulary_id.

icd10_rows = diag[diag['icd10'].notna()][['studyid', 'icd10']].copy()
icd10_rows.columns = ['person_id', 'ICD']
icd10_rows['vocabulary_id'] = 'ICD10CM'

icd9_rows = diag[diag['icd9'].notna()][['studyid', 'icd9']].copy()
icd9_rows.columns = ['person_id', 'ICD']
icd9_rows['vocabulary_id'] = 'ICD9CM'

icd_for_phetk = pd.concat([icd10_rows, icd9_rows], ignore_index=True)
icd_for_phetk['ICD'] = icd_for_phetk['ICD'].astype(str).str.strip()

print(f'\nPheTK-format ICD table: {len(icd_for_phetk):,} rows')
print(f'  ICD-10 rows: {len(icd10_rows):,}')
print(f'  ICD-9 rows:  {len(icd9_rows):,}')

# Save to temp CSV (PheTK reads from file)
phetk_icd_path = OUTPUT_DIR / 'temp_icd_for_phetk.csv'
icd_for_phetk.to_csv(phetk_icd_path, index=False)
print(f'  Saved to: {phetk_icd_path}')

In [ ]:
# ── 4c. Initialize PheTK and load PheCode X mapping table ──
from PheTK.Phecode import Phecode
from PheTK._utils import get_phecode_mapping_table

# Initialize PheTK with custom platform — this validates the ICD data format
phetk = Phecode(platform="custom", icd_df_path=str(phetk_icd_path))

print(f'\nPheTK loaded {phetk.icd_events.shape[0]:,} ICD events')
print(f'Flag distribution:')
print(phetk.icd_events['flag'].value_counts())

# Load PheTK's built-in PheCode X mapping table (156k ICD→PheCode entries)
# This is the same table PheTK uses internally in count_phecode()
phecode_map = get_phecode_mapping_table(
    phecode_version="X",
    icd_version="US",
    phecode_map_file_path=None,
    keep_all_columns=True
)
print(f'\nPheTK PheCode X mapping: {phecode_map.shape[0]:,} entries')
print(f'  Unique PheCode IDs: {phecode_map["phecode"].n_unique():,}')
print(f'  Unique phenotype strings: {phecode_map["phecode_string"].n_unique():,}')
print(f'  Columns: {phecode_map.columns}')

In [ ]:
# ── 4d. Map each diagnosis row to a PheCode phenotype string ──
# PheTK's count_phecode() aggregates to patient-level counts, but BEHRT needs
# per-encounter phenotypes. So we use PheTK's mapping table to join at the
# encounter level ourselves.
#
# Strategy for each row:
#   1. Try ICD-10 code → PheCode (inner join on ICD + flag=10)
#   2. If no match, try ICD-9 code → PheCode (inner join on ICD + flag=9)
# Both coding systems are treated as real diagnoses with equal weight.

import polars as pl

# Build lookup dicts from PheTK's mapping table (keyed on (ICD, flag) → phecode_string)
phecode_map_pd = phecode_map.to_pandas()
icd_to_phenotype = {}
for _, row in phecode_map_pd.iterrows():
    key = (row['ICD'], int(row['flag']))
    icd_to_phenotype[key] = row['phecode_string']

print(f'Built lookup with {len(icd_to_phenotype):,} (ICD, flag) → phenotype entries')

def map_row_to_phecode(row):
    """
    Map a diagnosis row to a PheCode phenotype string using PheTK's mapping.
    Tries ICD-10 first (flag=10), then ICD-9 (flag=9).
    """
    if pd.notna(row['icd10']):
        code = str(row['icd10']).strip()
        phe = icd_to_phenotype.get((code, 10))
        if phe is not None:
            return phe
    if pd.notna(row['icd9']):
        code = str(row['icd9']).strip()
        phe = icd_to_phenotype.get((code, 9))
        if phe is not None:
            return phe
    return None

print('Mapping ICD codes to PheCode phenotypes via PheTK mapping...')
diag['phenotype'] = diag.apply(map_row_to_phecode, axis=1)

n_mapped = diag['phenotype'].notna().sum()
n_unmapped = diag['phenotype'].isna().sum()
total = len(diag)
print(f'\nMapping results:')
print(f'  Mapped:   {n_mapped:,} / {total:,} ({100*n_mapped/total:.1f}%)')
print(f'  Unmapped: {n_unmapped:,} / {total:,} ({100*n_unmapped/total:.1f}%)')
print(f'  Unique phenotypes in data: {diag["phenotype"].nunique():,}')

In [ ]:
# ── 4e. Inspect what's being dropped ──
unmapped_rows = diag[diag['phenotype'].isna()]
if len(unmapped_rows) > 0:
    print('Most common unmapped ICD-10 codes (typically Z-codes, IMO codes, etc.):')
    top_unmapped = unmapped_rows['icd10'].value_counts().head(15)
    for code, count in top_unmapped.items():
        print(f'  {str(code):12s}  ({count:,} rows)')

    print(f'\nMost common unmapped ICD-9 codes (rows where ICD-10 was also unmapped):')
    icd9_only_unmapped = unmapped_rows[unmapped_rows['icd10'].isna()]
    if len(icd9_only_unmapped) > 0:
        top_unmapped_9 = icd9_only_unmapped['icd9'].value_counts().head(10)
        for code, count in top_unmapped_9.items():
            print(f'  {str(code):12s}  ({count:,} rows)')

# ── 4f. Drop unmapped rows and rows with missing dates ──
before = len(diag)
diag = diag.dropna(subset=['contact_dt', 'phenotype']).copy()
print(f'\nDropped {before - len(diag):,} rows (missing date or unmappable ICD code)')
print(f'Remaining: {len(diag):,} diagnosis rows')
print(f'Unique phenotypes: {diag["phenotype"].nunique():,}')

In [ ]:
# ── 4g. Merge birth_year from demographics ──
# Drop any existing birth_year column first to avoid _x/_y suffixes on re-runs
if 'birth_year' in diag.columns:
    diag = diag.drop(columns=['birth_year'])
for col in [c for c in diag.columns if c.startswith('birth_year_')]:
    diag = diag.drop(columns=[col])

birth_lookup = demo[['studyid', 'birth_year']].copy()
diag = diag.merge(birth_lookup, on='studyid', how='left')

n_no_birth = diag['birth_year'].isna().sum()
print(f'Diagnoses rows with no birth_year (patient not in demographics): {n_no_birth:,}')

if n_no_birth > 0:
    diag = diag.dropna(subset=['birth_year']).copy()
    print(f'After dropping: {len(diag):,} rows remain')

# ── 4h. Compute age at each contact ──
diag['age_at_contact'] = diag['contact_dt'].dt.year - diag['birth_year'].astype(int)

# Clamp ages to valid range [0, 119] for BEHRT's age vocabulary
diag['age_at_contact'] = diag['age_at_contact'].clip(0, 119).astype(int)

print(f'\nAge at contact range: {diag["age_at_contact"].min()} – {diag["age_at_contact"].max()}')

display(diag[['studyid', 'contact_dt', 'icd10', 'icd9', 'phenotype', 'birth_year', 'age_at_contact']].head(5))

---
## Step 5: Build Patient Sequences

This is the core transformation. For each patient, we:
1. Sort all diagnosis rows by `contact_dt`
2. Group diagnoses that occur on the **same day** into one "visit"
3. Insert a `'SEP'` token between visits (and after the last visit)
4. Build parallel `code` and `age` lists

The `code` list now contains **PheCode phenotype strings** (not raw ICD codes),
so an ICD-9 `714.0` and an ICD-10 `M06.9` both appear as `"Rheumatoid arthritis"`.

This mirrors the logic in `embeddings_BEHRT/building_patients_seqs.py`, adapted
for UChicago's date-based (rather than integer-day-offset) format.

In [ ]:
from tqdm import tqdm
tqdm.pandas()

def build_patient_sequence(patient_df):
    """
    Build a BEHRT-compatible sequence from a single patient's diagnosis rows.
    
    Input:
        patient_df: DataFrame grouped by studyid, with columns:
            - contact_dt: datetime of the encounter
            - phenotype: PheCode phenotype string (mapped from ICD-9/10)
            - age_at_contact: integer age at this encounter
    
    Output:
        pd.Series with:
            - 'code': list of phenotype strings with 'SEP' between visits
            - 'age': list of age strings (parallel to code)
    
    The SEP token marks visit boundaries. Within a visit (same contact_dt),
    multiple phenotypes share the same age. Between visits, a SEP is
    inserted with the age of the preceding visit.
    
    Duplicate phenotypes within the same visit are kept (same phenotype
    from different dx_ids may carry clinical significance).
    """
    # Sort by date, then by phenotype for deterministic ordering within a day
    patient_df = patient_df.sort_values(['contact_dt', 'phenotype'])
    
    code_seq = []
    age_seq = []
    prev_date = None
    
    for _, row in patient_df.iterrows():
        current_date = row['contact_dt'].date()
        age_str = str(row['age_at_contact'])
        
        # Insert SEP when the date changes (new visit)
        if prev_date is not None and current_date != prev_date:
            code_seq.append('SEP')
            age_seq.append(age_seq[-1])  # SEP gets the age of the previous visit
        
        code_seq.append(row['phenotype'])
        age_seq.append(age_str)
        prev_date = current_date
    
    # Trailing SEP to close the last visit
    if code_seq:
        code_seq.append('SEP')
        age_seq.append(age_seq[-1])
    
    return pd.Series({'code': code_seq, 'age': age_seq})


print('Building patient sequences (this may take a few minutes)...')
sequences = diag.groupby('studyid').progress_apply(build_patient_sequence)
sequences = sequences.reset_index()

print(f'\nBuilt sequences for {len(sequences):,} patients')

In [ ]:
# ── Drop patients with empty sequences ──
before_drop = len(sequences)
sequences = sequences[sequences['code'].apply(len) > 0].reset_index(drop=True)
print(f'Dropped {before_drop - len(sequences):,} patients with empty sequences')
print(f'Remaining: {len(sequences):,} patients')

# ── Sequence length statistics ──
seq_lengths = sequences['code'].apply(len)
print(f'\nSequence length stats:')
print(f'  Min:    {seq_lengths.min()}')
print(f'  Median: {seq_lengths.median():.0f}')
print(f'  Mean:   {seq_lengths.mean():.1f}')
print(f'  Max:    {seq_lengths.max()}')
print(f'  Std:    {seq_lengths.std():.1f}')

# Percentage that exceed BEHRT's max_len=512
n_over_512 = (seq_lengths > 512).sum()
print(f'\n  Sequences > 512 tokens (will be truncated by BEHRT): {n_over_512:,} ({100*n_over_512/len(sequences):.1f}%)')

In [ ]:
# ── Age distribution across patient sequences ──
# Each patient's age list contains the age at every token (including SEPs).
# We extract the min and max age per patient to understand the age span,
# then bin all ages to see the overall distribution.

import collections

all_ages = []
patient_age_ranges = []
for _, row in sequences.iterrows():
    ages_int = [int(a) for a in row['age']]
    all_ages.extend(ages_int)
    patient_age_ranges.append((min(ages_int), max(ages_int)))

all_ages_arr = np.array(all_ages)

# Define age bins
bins = [0, 20, 30, 40, 50, 60, 70, 80, 90, 100]
labels = ['<20', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90+']
age_binned = pd.cut(all_ages_arr, bins=bins, labels=labels, right=False)

print('=== Age distribution across all tokens in sequences ===')
bin_counts = age_binned.value_counts().sort_index()
for label, count in bin_counts.items():
    bar = '#' * int(50 * count / bin_counts.max())
    print(f'  {label:6s}: {count:>10,} tokens  ({100*count/len(all_ages_arr):5.1f}%)  {bar}')
print(f'  Total:  {len(all_ages_arr):>10,} tokens')

# Per-patient: age at first and last token
min_ages = np.array([r[0] for r in patient_age_ranges])
max_ages = np.array([r[1] for r in patient_age_ranges])
print(f'\n=== Per-patient age range ===')
print(f'  Youngest age at first token: {min_ages.min()}')
print(f'  Oldest age at last token:    {max_ages.max()}')
print(f'  Median entry age:            {np.median(min_ages):.0f}')
print(f'  Median exit age:             {np.median(max_ages):.0f}')

In [ ]:
# ── Inspect one patient's sequence ──
sample_idx = 0
sample = sequences.iloc[sample_idx]
print(f'Patient studyid={sample["studyid"]}')
print(f'Sequence length: {len(sample["code"])} tokens')
print(f'\nFirst 30 tokens (code):  {sample["code"][:30]}')
print(f'First 30 tokens (age):   {sample["age"][:30]}')

---
## Step 6: Rename `studyid` → `enrolid`

BEHRT's `MLMLoader` and `InferenceDataset` look for the column `enrolid`
when using a `sex_dict`. We rename to stay compatible without modifying
any BEHRT source code.

In [ ]:
sequences = sequences.rename(columns={'studyid': 'enrolid'})
print(f'Columns in sequences DataFrame: {list(sequences.columns)}')
print(f'Sample enrolid values: {sequences["enrolid"].head(5).tolist()}')

---
## Step 7: Truncate Sequences & Assign Risk Class Labels

The fine-tuned BEHRT LoRA model predicts a **4-class time-to-hip-fracture risk**:

| Class | Meaning | Days from DM onset |
|-------|---------|---------------------|
| 0 | No fracture / >10 years | NaN, ≤0, or >3650 |
| 1 | 1–5 years | 366–1825 |
| 2 | Within 1 year (highest risk) | 1–365 |
| 3 | 5–10 years | 1826–3650 |

To match the original fine-tuning pipeline (`build_finetune_dataset.py` + `building_dataset_for_hip_within_1_year.ipynb`):

1. **Identify** which patients had a confirmed hip fracture (`nt_fx_hip == 1` from the outcome file)
2. **Find** the date of the first fracture phenotype (`'Fracture of femur'` or `'Pathological fracture'`) for each patient
3. **Truncate** every patient's sequence *before* the first fracture phenotype token — even non-hip-fracture patients get truncated if they have these codes
4. **Compute** `days_to_hip_frac = first_fracture_date − start_dat` for confirmed hip-fracture patients
5. **Assign** `risk_class` using the same thresholds as the original pipeline
6. Patients with **no confirmed hip fracture** (including those with non-hip fracture phenotypes) → class 0

In [ ]:
# ── 7a. Load outcome data and find first fracture dates ──

outcome = pd.read_csv(DATA_DIR / 'Anony DM Outcome and FRAX.csv')
hip_fx_ids = set(outcome[outcome['nt_fx_hip'] == 1]['studyid'])
print(f'Confirmed hip fracture patients (nt_fx_hip=1): {len(hip_fx_ids)}')
print(f'Total patients in outcome file: {len(outcome):,}')

HIP_FRACTURE_PHENOTYPES = {'Fracture of femur', 'Pathological fracture'}

# Find the first fracture phenotype date per patient from the diagnosis data.
# We need the actual dates, so we go back to the raw 'diag' DataFrame which
# already has 'contact_dt' (parsed) and 'phenotype' columns from Step 4.
fx_diag = diag[diag['phenotype'].isin(HIP_FRACTURE_PHENOTYPES)].copy()
first_fx_per_patient = fx_diag.groupby('studyid')['contact_dt'].min().reset_index()
first_fx_per_patient.columns = ['studyid', 'first_fx_date']

# Merge with start_dat_parsed to compute time-to-fracture (defined in Step 3)
first_fx_per_patient = first_fx_per_patient.merge(
    demo[['studyid', 'start_dat_parsed']], on='studyid', how='left'
)
first_fx_per_patient['days_to_hip_frac'] = (
    first_fx_per_patient['first_fx_date'] - first_fx_per_patient['start_dat_parsed']
).dt.days

print(f'\nPatients with fracture phenotypes in diagnoses: {len(first_fx_per_patient)}')
print(f'  Of which confirmed hip fracture (nt_fx_hip=1): {len(first_fx_per_patient[first_fx_per_patient["studyid"].isin(hip_fx_ids)])}')
print(f'  Of which non-hip fracture (nt_fx_hip=0):       {len(first_fx_per_patient[~first_fx_per_patient["studyid"].isin(hip_fx_ids)])}')

print(f'\nTime-to-fracture stats for confirmed hip fx patients (days):')
hip_only = first_fx_per_patient[first_fx_per_patient['studyid'].isin(hip_fx_ids)]
print(hip_only['days_to_hip_frac'].describe())

In [ ]:
# ── 7b. Truncate sequences before hip fracture phenotypes ──
# Exactly mirrors build_finetune_dataset.py: find the first occurrence of
# any hip fracture phenotype and keep only the tokens strictly before it.
# Patients without these phenotypes keep their full sequences.

def truncate_before_hip(code_seq, age_seq):
    """Return (code, age) truncated before the first hip fracture phenotype."""
    for i, token in enumerate(code_seq):
        if token in HIP_FRACTURE_PHENOTYPES:
            return code_seq[:i], age_seq[:i]
    return code_seq, age_seq

truncated_codes = []
truncated_ages  = []

patients_truncated = 0
for _, row in tqdm(sequences.iterrows(), total=len(sequences), desc='Truncating'):
    c, a = truncate_before_hip(row['code'], row['age'])
    if len(c) < len(row['code']):
        patients_truncated += 1
    truncated_codes.append(c)
    truncated_ages.append(a)

sequences['code'] = truncated_codes
sequences['age']  = truncated_ages

print(f'Patients whose sequences were truncated: {patients_truncated}')

# Drop patients left with empty sequences after truncation
before_drop = len(sequences)
sequences = sequences[sequences['code'].apply(len) > 0].reset_index(drop=True)
dropped = before_drop - len(sequences)
print(f'Dropped {dropped} patients with empty post-truncation sequences')
print(f'Remaining: {len(sequences):,} patients')

In [ ]:
# ── 7c. Assign risk_class labels ──
# Thresholds match building_dataset_for_hip_within_1_year.ipynb exactly:
#   Class 0: no fracture, fracture ≤0 days (pre-existing), or >3650 days (>10yr)
#   Class 2: 1–365 days   (within 1 year — highest risk)
#   Class 1: 366–1825 days (1–5 years)
#   Class 3: 1826–3650 days (5–10 years)

def assign_risk_class(days):
    """Assign risk class from days-to-hip-fracture, matching original thresholds."""
    if pd.isna(days) or days <= 0 or days > 365 * 10:
        return 0
    elif days <= 365:
        return 2
    elif days <= 365 * 5:
        return 1
    elif days <= 365 * 10:
        return 3
    else:
        return 0

# Build a lookup: studyid → risk_class (only for confirmed hip fracture patients)
risk_lookup = {}
for _, row in first_fx_per_patient.iterrows():
    sid = row['studyid']
    if sid in hip_fx_ids:
        risk_lookup[sid] = assign_risk_class(row['days_to_hip_frac'])
    else:
        risk_lookup[sid] = 0  # non-hip fracture patients

# Assign risk_class: confirmed hip fx → computed class, everyone else → 0
sequences['risk_class'] = sequences['enrolid'].map(risk_lookup).fillna(0).astype(np.int8)

print('Risk class distribution:')
print(sequences['risk_class'].value_counts().sort_index())
print(f'\nTotal patients: {len(sequences):,}')
print(f'  Class 0 (no fx / >10yr): {(sequences["risk_class"] == 0).sum():,}')
print(f'  Class 1 (1-5 years):     {(sequences["risk_class"] == 1).sum():,}')
print(f'  Class 2 (<1 year):       {(sequences["risk_class"] == 2).sum():,}')
print(f'  Class 3 (5-10 years):    {(sequences["risk_class"] == 3).sum():,}')

In [ ]:
# ── 7d. Sanity checks ──

# 1. Verify no hip fracture codes remain in ANY sequence
print('Verifying no hip fracture phenotypes remain in sequences...')
leaked = sequences['code'].apply(
    lambda seq: any(t in HIP_FRACTURE_PHENOTYPES for t in seq)
).sum()
print(f'  Sequences still containing fracture codes: {leaked}')
assert leaked == 0, f'DATA LEAK: {leaked} sequences still contain fracture codes!'
print('  ✓ No data leakage')

# 2. All confirmed hip fracture patients should have a non-zero class
# (unless their fracture was pre-existing / >10yr)
hip_fx_in_seq = sequences[sequences['enrolid'].isin(hip_fx_ids)]
class_0_hip_fx = (hip_fx_in_seq['risk_class'] == 0).sum()
print(f'\nConfirmed hip fx patients in final dataset: {len(hip_fx_in_seq)}')
print(f'  With class 0 (pre-existing or >10yr): {class_0_hip_fx}')
print(f'  With class 1 (1-5yr): {(hip_fx_in_seq["risk_class"] == 1).sum()}')
print(f'  With class 2 (<1yr):  {(hip_fx_in_seq["risk_class"] == 2).sum()}')
print(f'  With class 3 (5-10yr): {(hip_fx_in_seq["risk_class"] == 3).sum()}')

# 3. Post-truncation sequence length stats
seq_lengths = sequences['code'].apply(len)
print(f'\nPost-truncation sequence length stats:')
print(f'  Min:    {seq_lengths.min()}')
print(f'  Median: {seq_lengths.median():.0f}')
print(f'  Mean:   {seq_lengths.mean():.1f}')
print(f'  Max:    {seq_lengths.max()}')

# 4. Final DataFrame schema
print(f'\nFinal sequences DataFrame:')
print(f'  Shape: {sequences.shape}')
print(f'  Columns: {list(sequences.columns)}')
print(f'  dtypes: {dict(sequences.dtypes)}')

---
## Step 8: Load the Original BEHRT Vocabularies

Since we are **validating** the existing BEHRT model (not retraining), we must use
the **same vocabularies** the model was trained with. This ensures that each phenotype
string maps to the same learned embedding.

- The original `token2idx` has **3,304 phenotype tokens** + 5 special tokens (PAD, CLS, SEP, MASK, UNK)
- The original `age2idx` maps ages 0–119 + PAD + UNK → 122 entries
- UChicago phenotypes not in the original vocab will map to `UNK` at inference time
- We check coverage below to quantify how much UChicago data is representable

In [ ]:
# ── 8a. Load the original BEHRT token vocabulary ──
with open(BEHRT_DIR / 'token2idx.pkl', 'rb') as f:
    token2idx = pickle.load(f)

print(f'Original BEHRT token2idx: {len(token2idx):,} entries')
print(f'  Special tokens: PAD={token2idx["PAD"]}, CLS={token2idx["CLS"]}, SEP={token2idx["SEP"]}, MASK={token2idx["MASK"]}, UNK={token2idx["UNK"]}')
print(f'  Phenotype tokens: {len(token2idx) - 5:,}')
print(f'\nSample phenotype entries:')
for token in list(token2idx.keys())[5:15]:
    print(f'  {token!r:40s} → {token2idx[token]}')

# ── 7b. Load the original BEHRT age vocabulary ──
with open(BEHRT_DIR / 'age2idx.pkl', 'rb') as f:
    age2idx = pickle.load(f)

print(f'\nOriginal BEHRT age2idx: {len(age2idx)} entries')
print(f'  PAD→{age2idx["PAD"]}, UNK→{age2idx["UNK"]}, "0"→{age2idx["0"]}, "65"→{age2idx["65"]}, "119"→{age2idx["119"]}')

# ── 7c. Check coverage: how many UChicago phenotypes are in the vocab? ──
uchicago_phenotypes = set()
for code_list in sequences['code']:
    uchicago_phenotypes.update(code_list)
uchicago_phenotypes.discard('SEP')

in_vocab = uchicago_phenotypes & set(token2idx.keys())
oov = uchicago_phenotypes - set(token2idx.keys())

print(f'\n=== UChicago phenotype coverage in original BEHRT vocab ===')
print(f'  Unique phenotypes in UChicago data: {len(uchicago_phenotypes):,}')
print(f'  In original BEHRT vocab:            {len(in_vocab):,} ({100*len(in_vocab)/len(uchicago_phenotypes):.1f}%)')
print(f'  Out-of-vocabulary (will be UNK):    {len(oov):,} ({100*len(oov)/len(uchicago_phenotypes):.1f}%)')

if oov:
    print(f'\n  OOV phenotypes:')
    for p in sorted(oov)[:20]:
        print(f'    "{p}"')
    if len(oov) > 20:
        print(f'    ... and {len(oov) - 20} more')

# Row-level: what fraction of tokens in sequences will be UNK?
total_tokens = 0
unk_tokens = 0
for code_list in sequences['code']:
    for c in code_list:
        if c == 'SEP':
            continue
        total_tokens += 1
        if c not in token2idx:
            unk_tokens += 1
print(f'\n  Token-level: {unk_tokens:,} / {total_tokens:,} tokens will be UNK ({100*unk_tokens/total_tokens:.2f}%)')

---
## Step 9: Build the Sex Dictionary

BEHRT's `MLMLoader` and `InferenceDataset` accept an optional `sex_dict` mapping
`enrolid → int` using the same convention as the original training data:
- `0` = PAD / unknown
- `1` = male
- `2` = female


In [ ]:
sex_map = {'male': 1, 'Male': 1, 'female': 2, 'Female': 2}

sex_dict = {}
for _, row in demo.iterrows():
    sid = row['studyid']
    sex_val = sex_map.get(row['sex'], 0)
    sex_dict[sid] = sex_val

n_male = sum(1 for v in sex_dict.values() if v == 1)
n_female = sum(1 for v in sex_dict.values() if v == 2)
n_unknown = sum(1 for v in sex_dict.values() if v == 0)

print(f'Sex dictionary: {len(sex_dict):,} patients')
print(f'  Male:    {n_male:,}')
print(f'  Female:  {n_female:,}')
print(f'  Unknown: {n_unknown:,}')

n_missing_sex = sum(1 for eid in sequences['enrolid'] if eid not in sex_dict)
print(f'\nPatients in sequences missing from sex_dict: {n_missing_sex}')

---
## Step 10: Save BEHRT-Ready Artifacts

We save the **UChicago-specific** artifacts. The sequences now include
`risk_class` labels (0–3) from Step 7, making them directly compatible with
the fine-tuned BEHRT LoRA model. The token and age vocabularies come from the
original BEHRT model and are loaded directly at inference time.

| File | Contents | Used by |
|------|----------|--------|
| `uchicago_dm_patient_sequences.pkl` | DataFrame with `enrolid`, `code`, `age` | `MLMLoader`, `InferenceDataset` |
| `uchicago_dm_sex_dict.pkl` | `dict[int, int]` (enrolid → sex) | `MLMLoader` (optional) |


In [ ]:
# ── Save patient sequences (now includes risk_class) ──
seq_path = OUTPUT_DIR / 'uchicago_dm_patient_sequences.pkl'
sequences.to_pickle(seq_path)
print(f'Saved: {seq_path}')
print(f'  Shape: {sequences.shape}')
print(f'  Columns: {list(sequences.columns)}')
print(f'  risk_class distribution: {dict(sequences["risk_class"].value_counts().sort_index())}')

# ── Save sex dictionary ──
sex_path = OUTPUT_DIR / 'uchicago_dm_sex_dict.pkl'
with open(sex_path, 'wb') as f:
    pickle.dump(sex_dict, f)
print(f'\nSaved: {sex_path}')
print(f'  Entries: {len(sex_dict):,}')

# ── Clean up temp file ──
temp_file = OUTPUT_DIR / 'temp_icd_for_phetk.csv'
if temp_file.exists():
    temp_file.unlink()
    print(f'\nCleaned up temp file: {temp_file}')

print(f'\nNote: token2idx and age2idx are loaded from the original BEHRT model at:')
print(f'  {BEHRT_DIR / "token2idx.pkl"}')
print(f'  {BEHRT_DIR / "age2idx.pkl"}')


---
## Step 11: Validation — End-to-End Smoke Test

We reload the saved artifacts and run them through BEHRT's `MLMLoader`
to verify that everything works end-to-end. This catches:
- Vocabulary mismatches (codes in sequences not in `token2idx`)
- Age values outside the age vocabulary
- Shape / tensor errors in the data loader
- Sex dictionary lookup failures

In [ ]:
import sys
sys.path.insert(0, str(BEHRT_DIR))
from MLM import MLMLoader
import torch

# ── Reload UChicago artifacts ──
seq_reload = pd.read_pickle(OUTPUT_DIR / 'uchicago_dm_patient_sequences.pkl')
with open(OUTPUT_DIR / 'uchicago_dm_sex_dict.pkl', 'rb') as f:
    sex_reload = pickle.load(f)

# ── Load original BEHRT vocabularies ──
with open(BEHRT_DIR / 'token2idx.pkl', 'rb') as f:
    tok_reload = pickle.load(f)
with open(BEHRT_DIR / 'age2idx.pkl', 'rb') as f:
    age_reload = pickle.load(f)

print(f'Reloaded sequences: {len(seq_reload):,} patients')
print(f'Original BEHRT token vocab: {len(tok_reload):,} tokens')
print(f'Original BEHRT age vocab: {len(age_reload)} entries')
print(f'Reloaded sex dict: {len(sex_reload):,} entries')


In [ ]:
# ── Create MLMLoader and fetch one batch ──
MAX_LEN = 512

# Use a small subset for the smoke test
test_subset = seq_reload.head(100).reset_index(drop=True)

dataset = MLMLoader(
    dataframe=test_subset,
    token2idx=tok_reload,
    age2idx=age_reload,
    max_len=MAX_LEN,
    sex_dict=sex_reload,
    code='code',
    age='age'
)

print(f'Dataset size: {len(dataset)}')

# Fetch one sample and inspect tensor shapes
age_t, code_t, pos_t, seg_t, mask_t, label_t, sex_t = dataset[0]

print(f'\n=== Single sample tensor shapes ===')
print(f'  age:      {age_t.shape}  dtype={age_t.dtype}')
print(f'  code:     {code_t.shape}  dtype={code_t.dtype}')
print(f'  position: {pos_t.shape}  dtype={pos_t.dtype}')
print(f'  segment:  {seg_t.shape}  dtype={seg_t.dtype}')
print(f'  mask:     {mask_t.shape}  dtype={mask_t.dtype}')
print(f'  labels:   {label_t.shape}  dtype={label_t.dtype}')
print(f'  sex:      {sex_t.shape}  dtype={sex_t.dtype}')

# Verify all tensor lengths equal MAX_LEN
assert all(t.shape[0] == MAX_LEN for t in [age_t, code_t, pos_t, seg_t, mask_t, label_t, sex_t]), \
    'Tensor length mismatch!'
print(f'\nAll tensors have length {MAX_LEN}. Smoke test PASSED.')

In [ ]:
# ── Test with DataLoader (batched) ──
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=16, shuffle=False)
batch = next(iter(loader))

print('=== Batch tensor shapes ===')
names = ['age', 'code', 'position', 'segment', 'mask', 'labels', 'sex']
for name, tensor in zip(names, batch):
    print(f'  {name:10s}: {tensor.shape}')

print(f'\nDataLoader batch test PASSED.')

---
## Step 12: Verify Model Compatibility

As a final check, we instantiate a `BehrtForMaskedLM` model with a config
derived from our UChicago vocabularies, run one forward pass on the test
batch, and confirm we get a valid loss.

In [ ]:
from MLM_model import BehrtConfig, BehrtForMaskedLM

config = BehrtConfig(
    vocab_size=len(tok_reload),
    hidden_size=288,                    # same as BEHRT default
    num_hidden_layers=6,
    num_attention_heads=12,
    intermediate_size=512,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    max_position_embeddings=MAX_LEN,
    seg_vocab_size=2,
    age_vocab_size=len(age_reload),
    sex_vocab_size=3,
)

model = BehrtForMaskedLM(config)
model.eval()

param_count = sum(p.numel() for p in model.parameters())
print(f'Model instantiated with {param_count:,} parameters')
print(f'  vocab_size:     {config.vocab_size}')
print(f'  age_vocab_size: {config.age_vocab_size}')
print(f'  sex_vocab_size: {config.sex_vocab_size}')
print(f'  hidden_size:    {config.hidden_size}')
print(f'  max_position:   {config.max_position_embeddings}')

# Forward pass
age_b, code_b, pos_b, seg_b, mask_b, label_b, sex_b = batch

with torch.no_grad():
    loss, logits, labels_out = model(
        input_ids=code_b,
        age_ids=age_b,
        seg_ids=seg_b,
        posi_ids=pos_b,
        sex_ids=sex_b,
        attention_mask=mask_b,
        masked_lm_labels=label_b
    )

print(f'\nForward pass successful!')
print(f'  MLM loss: {loss.item():.4f}')
print(f'  Logits shape: {logits.shape}')
print(f'\nUChicago DM data is fully BEHRT-compatible.')

---
## Summary

### What was done

| Step | Action | Result |
|------|--------|--------|
| 1 | Loaded raw CSVs | 13,551 patients (demographics), ~3.4M diagnosis rows |
| 2 | Quality checks | Verified ID overlap, missing values, ICD coverage |
| 3 | Derived birth year | `birth_year = year_of(start_dat) - int(age_start_dat)` |
| 4 | **ICD → PheCode mapping via PheTK** | Used `PheTK.Phecode` with PheCode X to map ICD-9 and ICD-10 to shared phenotype strings |
| 5 | Built sequences | Per-patient `code` + `age` lists with `SEP` separators (tokens are phenotype strings) |
| 6 | Renamed IDs | `studyid` → `enrolid` for BEHRT compatibility |
| 7 | **Truncation + risk class labeling** | Truncated sequences before fracture phenotypes; assigned 4-class `risk_class` (0/1/2/3) using same thresholds as original fine-tuning |
| 8 | **Loaded original BEHRT vocabs** | Reused trained `token2idx` (3,309 tokens) and `age2idx` (122 entries) — no new vocabs built |
| 9 | Sex dictionary | `male→1`, `female→2` per patient |
| 10 | Saved artifacts | 2 pickle files in `uchicago_output/` (sequences + sex dict) |
| 11 | Smoke test | `MLMLoader` produces correct tensor shapes |
| 12 | Model test | Forward pass through `BehrtForMaskedLM` yields valid loss |

### Risk class assignment (Step 7)

Matches `building_dataset_for_hip_within_1_year.ipynb` thresholds:

| Class | Meaning | Condition |
|-------|---------|-----------|
| 0 | No fracture / pre-existing / >10yr | `days_to_hip_frac` is NaN, ≤0, or >3650 |
| 1 | 1–5 years | 366 ≤ days ≤ 1825 |
| 2 | Within 1 year (highest risk) | 1 ≤ days ≤ 365 |
| 3 | 5–10 years | 1826 ≤ days ≤ 3650 |

- Ground truth: `nt_fx_hip == 1` from `Anony DM Outcome and FRAX.csv` (71 patients)
- Reference date: `start_dat` (DM cohort entry)
- Truncation: removes tokens at and after the first `'Fracture of femur'` or `'Pathological fracture'` for ALL patients

### Output files

All saved to `embeddings_BEHRT/uchicago_output/`:
- `uchicago_dm_patient_sequences.pkl` — patient data (enrolid, code, age, **risk_class**)
- `uchicago_dm_sex_dict.pkl` — sex lookup for each patient

The token and age vocabularies are **not** saved here — the original BEHRT
vocabs from `embeddings_BEHRT/token2idx.pkl` and `embeddings_BEHRT/age2idx.pkl`
are used directly.

### Next steps

The UChicago dataset now has the same structure as the Marketscan fine-tuning data:
truncated sequences with 4-class `risk_class` labels. To evaluate:

1. Load `uchicago_output/uchicago_dm_patient_sequences.pkl` with the original vocabs
2. Create a `FinetuneDataset` using `risk_class` as the label
3. Run inference with the fine-tuned BEHRT LoRA checkpoint
4. Compare predicted vs actual risk classes (accuracy, ROC-AUC per class, confusion matrix)
